In [45]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

In [46]:
df = pd.read_csv('../../DataBox/covid_gen.csv')

In [47]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'], test_size=0.2)

In [48]:
X_train.isnull().sum()

age       0
gender    0
fever     6
cough     0
city      0
dtype: int64

Without Column Transformer -> Hectic

In [49]:
# *************** Handling Each Type Of encoding Seperately **************

# adding simple imputer to fever column

# create SI object
si = SimpleImputer()

# Transform 
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.transform(X_test[['fever']])
                                 
                                 
                                 
# ************* Encoding ************** 

# Ordinal Encoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']],)

# also the test data
X_test_cough = oe.transform(X_test[['cough']])



# OneHot Encoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.transform(X_test[['gender','city']])


# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape


# Numpy Concat data in Float Form (Since Simple Imputer has Float Data type -> Everything Changed to Float)

# Club All the Data Together
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)


# Float to Integer (Saves Memory)

X_train_transformed = X_train_transformed.astype(np.int32)
X_test_transformed = X_test_transformed.astype(np.int32)

Using Column Transformer

In [50]:
# Create Object
# Parameters  -> List of columns To Transform, Remainder -> Remaining Column (drop, pass-through) 
# List of Column -> each Transformation In Touple. (name of transformer) - transformer object (type of transformation) - (coulmn)

# ! sparse_output -> True -> Stores Only 1 (Ignores all 0) -> Save Ram - Memory

transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [51]:
X_train_transformed = transformer.fit_transform(X_train)
X_test_transformed = transformer.transform(X_test)